In [ ]:
from ultralytics import YOLO
import torch
import cv2
import sys
import os
from roboflow import Roboflow

In [ ]:
!"{sys.executable}" -m pip install roboflow

In [ ]:
!"{sys.executable}" -m pip install ultralytics

In [ ]:
rf = Roboflow(api_key="<API_KEY>")
project = rf.workspace("<WORKSPACE_NAME>").project("<DATASET_NAME>")
version = project.version(2)
dataset = version.download("yolov8")

In [ ]:
device = (
    "0" if torch.cuda.is_available()
    else "cpu"
)
print(f"Using Device: {device}")

In [ ]:
model = YOLO("yolov8n.pt")

In [ ]:
results = model.train(
    data = f"{dataset.location}/data.yaml",
    epochs = 20,
    imgsz = 640,
    device = device,
    name = "hand_gesture_model",
    workers = 0  
)

In [ ]:
best_model_path = "runs/detect/hand_gesture_model/weights/best.pt"

In [ ]:
def run_camera(best_model_path):
    try:
        model = YOLO(best_model_path)
        cap = cv2.VideoCapture(0)
        if not cap.isOpened():
            raise Exception("Could not open webcam")
        print("Camera started")
        while True:
            success, frame = cap.read()
            if not success:
                print("Failed to read frame")
            gray_frame = cv2.cvtColor(frame,cv2.COLOR_BGR2GRAY)
            input_frame = cv2.cvtColor(gray_frame,cv2.COLOR_GRAY2BGR)
            result = model.predict(input_frame, conf = 0.5, verbose = False)
            annodated_frame = result[0].plot()
            cv2.imshow("YOLO prediction (press q to quit)",annodated_frame)
            if cv2.waitKey(1) & 0xFF == ord('q'):
                print("Quit signal received")
                break
    except Exception as e:
        print(e)
    finally:
        if 'cap' in locals() and cap.isOpened():
            cap.release()
        cv2.destroyAllWindows()

In [ ]:
run_camera(best_model_path)